In [7]:
import os
import pandas as pd
from pathlib import Path
import ffmpeg

def audio_exists(video_folder_path, output_dir):
    video_root = Path(video_folder_path)
    output_root = Path(output_dir)
    master_csv_path = output_root / "master_audio_exist.csv"

    output_root.mkdir(parents=True, exist_ok=True)


    processed_files = set()
    master_data = []

    if master_csv_path.exists():
        print("Found existing master CSV. Loading progress to skip completed videos...")
        try:
            df_existing = pd.read_csv(master_csv_path)
            processed_files = set(df_existing['video_path'].tolist())
            master_data = df_existing.to_dict('records')
        except Exception as e:
            print(f"Could not read existing CSV, starting fresh. Error: {e}")
    video_extensions = ["*.mp4", "*.mkv", "*.avi", "*.mov"]
    video_files = []
    for ext in video_extensions:
        video_files.extend(list(video_root.rglob(ext)))
    print(f"Found {len(video_files)} total videos. {len(processed_files)} already processed.")

    for i, video_path in enumerate(video_files, 1):
        video_path_str = str(video_path)
        filename = video_path.name
        filename_stem = video_path.stem

        if video_path_str in processed_files:
            continue
        print(f"[{i}/{len(video_files)}] {filename}")

        try:
            probe = ffmpeg.probe(video_path_str, select_streams='a')
            exist_bool = False
            
            if probe.get('streams'):
                exist_bool=True

            master_data.append({
                "filename": filename,
                "video_path": video_path_str,
                "exist_bool": exist_bool
            })
            df_master = pd.DataFrame(master_data)
            df_master.to_csv(master_csv_path, index=False, encoding='utf-8-sig')

        except Exception as e:
            print(f"❌ Error processing {filename}: {e}")
            continue
INPUT_FOLDER = r"D:\KRSL_OnlineSchool\KRSL_OnlineSchool_final_dataset\video"
OUTPUT_FOLDER = r"D:\KRSL_OnlineSchool\Audio_Exist_Outputs"

In [12]:
audio_exists(INPUT_FOLDER, OUTPUT_FOLDER)

Found existing master CSV. Loading progress to skip completed videos...
Found 4563 total videos. 4562 already processed.
[218/4563] ._01_02_1_01_02.mp4
❌ Error processing ._01_02_1_01_02.mp4: ffprobe error (see stderr output for detail)


In [14]:
output_root = Path(OUTPUT_FOLDER)
master_csv_path = output_root / "master_audio_exist.csv"
df = pd.read_csv(master_csv_path)

In [17]:
true_count = df['exist_bool'].sum()
print(true_count)

2753
